In [9]:
import requests
import time
import os 
import pandas as pd
import codecs
import re
import unicodedata
from pathlib import Path
from rapidfuzz import process, fuzz


In [2]:
df = pd.read_csv("/Users/daniel/Documents/DAW/Steam-x-Twitch-x-IGDB-analysis/data/raw/twich_history/Twitch_game_data.csv", encoding="utf-8", encoding_errors="replace")

In [3]:
def cp1252_fallback(exc):
    if isinstance(exc, UnicodeDecodeError):
        # If utf-8 fails, try decoding the bad byte with cp1252
        res = exc.object[exc.start:exc.end].decode("cp1252")
        return (res, exc.end)
    raise TypeError(f"cannot handle {exc}")

# Register our custom error handler
codecs.register_error("cp1252_fallback", cp1252_fallback)

# Load the file using the fallback
twitch = pd.read_csv("/Users/daniel/Documents/DAW/Steam-x-Twitch-x-IGDB-analysis/data/raw/twich_history/Twitch_game_data.csv", encoding="utf-8", encoding_errors="cp1252_fallback")

# Fix R-style unicode escapes (e.g. <U+00E9> for é)
twitch["Game"] = twitch["Game"].str.replace(r"<U\+([0-9A-Fa-f]{4})>", lambda m: chr(int(m.group(1), 16)), regex=True)

In [4]:
df


,Rank,Game,Month,Year,Hours_watched,Hours_streamed,Peak_viewers,Peak_channels,Streamers,Avg_viewers,Avg_channels,Avg_viewer_ratio
0,1,League of Legends,1,2016,94377226,1362044,530270,2903,129172,127021,1833,69.29
1,2,Counter-Strike: Global Offensive,1,2016,47832863,830105,372654,2197,120849,64378,1117,57.62
2,3,Dota 2,1,2016,45185893,433397,315083,1100,44074,60815,583,104.26
3,4,Hearthstone,1,2016,39936159,235903,131357,517,36170,53749,317,169.29
4,5,Call of Duty: Black Ops III,1,2016,16153057,1151578,71639,3620,214054,21740,1549,14.03
...,...,...,...,...,...,...,...,...,...,...,...,...
20995,196,Pokémon HeartGold/SoulSilver,9,2024,683902,10093,10600,42,1443,951,14,67.76
20996,197,Grounded,9,2024,683226,37779,39473,116,7076,950,52,18.08
20997,198,Ragnarok Online,9,2024,679965,28491,2539,75,1066,945,39,23.87
20998,199,Politics,9,2024,673799,16675,19746,210,957,937,23,40.41


In [5]:
steam = pd.read_csv("/Users/daniel/Documents/DAW/Steam-x-Twitch-x-IGDB-analysis/data/raw/steam_games/steam_kaggle_games_march2025_full.csv", encoding="utf-8", encoding_errors="cp1252_fallback")
steam

,appid,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent
0,730,Counter-Strike 2,2012-08-21,0,0.00,1,"For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...",NaN,...,879,5174,350,0,1212356,"{'FPS': 90857, 'Shooter': 65397, 'Multiplayer'...",86,8632939,82,96473
1,578080,PUBG: BATTLEGROUNDS,2017-12-21,0,0.00,0,"LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...","LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...",Play PUBG: BATTLEGROUNDS for free. Land on str...,NaN,...,0,0,0,0,616738,"{'Survival': 14838, 'Shooter': 12727, 'Battle ...",59,2513842,68,16720
2,570,Dota 2,2013-07-09,0,0.00,2,"The most-played game on Steam. Every day, mill...","The most-played game on Steam. Every day, mill...","Every day, millions of players worldwide enter...",“A modern multiplayer masterpiece.” 9.5/10 – D...,...,1536,898,892,0,555977,"{'Free to Play': 59933, 'MOBA': 20158, 'Multip...",81,2452595,80,29366
3,271590,Grand Theft Auto V Legacy,2015-04-13,17,0.00,0,"When a young street hustler, a retired bank ro...","When a young street hustler, a retired bank ro...",Grand Theft Auto V for PC offers players the o...,NaN,...,771,7101,74,0,117698,"{'Open World': 32644, 'Action': 23539, 'Multip...",87,1803832,92,17517
4,488824,Tom Clancy's Rainbow Six® Siege,2015-12-01,17,19.99,9,Edition Comparison Ultimate Edition The Tom Cl...,“One of the best first-person shooters ever ma...,"Tom Clancy's Rainbow Six® Siege is an elite, t...",NaN,...,0,0,0,0,0,"{'FPS': 8082, 'Multiplayer': 6139, 'Tactical':...",84,1168404,76,13017
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94943,2115110,Outrun Them,2024-02-29,0,3.99,0,The name of the game is literally the game. Ou...,The name of the game is literally the game. Ou...,It's all in the name.. outrun them.,NaN,...,0,0,0,0,0,"{'2D Platformer': 91, 'Runner': 85, 'Platforme...",-1,-1,-1,-1
94944,1174200,Lands of Pharaoh: Episode 1,2020-01-07,0,10.00,0,Join Our Discord Channel About the Game Briefl...,Briefly: Lands of Pharaoh is a third person vi...,Lands of Pharaoh is a multi-camera-angle based...,NaN,...,0,0,0,0,0,"{'Action': 43, 'Adventure': 41, 'Indie': 41, '...",-1,-1,-1,-1
94945,1160190,Player One,2019-10-10,0,1.99,0,Player One is a new 3D game where you take con...,Player One is a new 3D game where you take con...,"3D Game, Thirdperson Platformer",NaN,...,0,0,0,0,0,"{'Indie': 31, 'Early Access': 21}",-1,-1,-1,-1
94946,3380340,DragonRoad,2025-01-25,0,29.99,0,"This is an open world game on 3D, also a m...","This is an open world game on 3D, also a m...","This is an open world game on 3D, Also a missi...",NaN,...,0,0,0,0,0,"{'RPG': 117, 'Action-Adventure': 111, 'Strateg...",-1,-1,-1,-1


In [ ]:
# imports for the matching
import unicodedata
from rapidfuzz import process, fuzz

# roman numerals -> numbers, so "Final Fantasy VII" and "Final Fantasy 7" look the same
roman = {"ii": "2", "iii": "3", "iv": "4", "v": "5", "vi": "6", "vii": "7", "viii": "8", "ix": "9",
         "xi": "11", "xii": "12", "xiii": "13", "xiv": "14", "xv": "15", "xvi": "16"}

def clean_name(name):
    # makes game names easier to compare
    if pd.isna(name):
        return ""
    name = str(name).lower()
    name = re.sub(r"[™®©]", " ", name)                  # remove TM, R, C symbols
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))   # remove accents
    name = name.replace("&", " and ")
    name = re.sub(r"[^\w\s]", " ", name)                # remove punctuation
    name = re.sub(r"\s+", " ", name).strip()
    name = re.sub(r"^the ", "", name)                   # remove "the" at the start
    words = name.split()
    words = [roman.get(w, w) for w in words]
    return " ".join(words)

def get_numbers(name):
    # all numbers in a name, so "madden nfl 20" is not matched with "madden nfl 22"
    return re.findall(r"\d+", name)


In [ ]:
# steam: drop columns that we don't need
drop_cols = ["detailed_description", "about_the_game", "short_description", "reviews", "header_image",
             "support_url", "support_email", "notes", "packages", "screenshots", "movies"]
steam_small = steam.drop(columns=drop_cols)

# add clean name and release date as a real date
steam_small["clean_name"] = steam_small["name"].apply(clean_name)
steam_small["release"] = pd.to_datetime(steam_small["release_date"], errors="coerce")

# some games are in steam more than once -> keep the earliest release date
steam_small = steam_small[steam_small["clean_name"] != ""]
steam_small = steam_small.sort_values("release")
steam_small = steam_small.drop_duplicates(subset="clean_name", keep="first")
steam_small = steam_small.drop(columns="release")
print(steam_small.shape)


In [ ]:
# twitch: clean name
twitch["clean_name"] = twitch["Game"].apply(clean_name)

steam_names = steam_small["clean_name"].tolist()
steam_names_set = set(steam_names)

# for every twitch game find the steam name that fits (exact first, then fuzzy)
matches = {}    # twitch clean name -> (steam clean name, score)

for game in twitch["clean_name"].unique():
    if game == "":
        continue

    if game in steam_names_set:
        matches[game] = (game, 100)
        continue

    # fuzzy: look at the 5 best candidates
    best = process.extract(game, steam_names, scorer=fuzz.ratio, score_cutoff=93, limit=5)
    for steam_name, score, index in best:
        same_numbers = get_numbers(game) == get_numbers(steam_name)
        # short names need a higher score, otherwise "dreams" would match "dream"
        if len(game) < 12 and score < 96:
            continue
        if same_numbers:
            matches[game] = (steam_name, score)
            break

print("exact:", sum(1 for m in matches.values() if m[1] == 100))
print("fuzzy:", sum(1 for m in matches.values() if m[1] < 100))


In [ ]:
# add the matched steam name and score to twitch
twitch["steam_clean_name"] = twitch["clean_name"].apply(lambda x: matches[x][0] if x in matches else None)
twitch["steam_match_score"] = twitch["clean_name"].apply(lambda x: matches[x][1] if x in matches else None)

# left join: all twitch rows stay, steam columns are empty if there is no match
steam_small = steam_small.rename(columns={"clean_name": "steam_clean_name"})
df = twitch.merge(steam_small, how="left", on="steam_clean_name")
df = df.drop(columns=["clean_name", "steam_clean_name"])
df.shape


In [ ]:
# twitch games that are not in steam -> not_in_steam.csv (one row per game)
not_in_steam = df[df["appid"].isna()]
not_in_steam = not_in_steam.groupby("Game").agg(months=("Month", "count"),
                                                total_hours_watched=("Hours_watched", "sum"))
not_in_steam = not_in_steam.sort_values("total_hours_watched", ascending=False).reset_index()
not_in_steam.to_csv("/Users/daniel/Documents/DAW/Steam-x-Twitch-x-IGDB-analysis/data/processed/not_in_steam.csv", index=False)
print(len(not_in_steam), "games not in steam")
not_in_steam.head()


In [ ]:
# keep only the rows that matched a steam game
df = df[df["appid"].notna()].reset_index(drop=True)
df.to_csv("/Users/daniel/Documents/DAW/Steam-x-Twitch-x-IGDB-analysis/data/processed/twitch_steam_combined.csv", index=False)
print(df.shape, df["Game"].nunique(), "games")
df.head()
